# Assignment 1

**Name:** Nazia Peer  
**Course:** Deploying AI  


## Objectives

1. Structured summary generation using a non–GPT-5 model.
2. Automated evaluation using DeepEval.
3. A feedback-driven enhancement loop.
4. Critical reflection on reliability and control limitations.


## 1. Environment Configuration

This notebook uses the course API gateway.

Secrets are loaded from a local `.secrets` file using dotenv.  
No API keys are hard-coded.

In [1]:


# Load secrets
%load_ext dotenv
%dotenv ../05_src/.secrets

import os

print("API_GATEWAY_KEY loaded:", bool(os.getenv("API_GATEWAY_KEY")))

API_GATEWAY_KEY loaded: True


In [2]:
from openai import OpenAI

client = OpenAI(
    base_url="https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1",
    api_key="any-value",  # gateway uses header instead
    default_headers={"x-api-key": os.getenv("API_GATEWAY_KEY", "")},
)

## 2. Document Selection and Loading

The article *Managing Oneself* by Peter Drucker is used.

This document:
- Contains a clear thesis.
- Includes multiple conceptual dimensions.
- Allows evaluation of coverage and faithfulness.

In [3]:
import requests
from pathlib import Path
import re

documents_path = Path("../../05_src/documents")
documents_path.mkdir(parents=True, exist_ok=True)

pdf_url = "https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf"
pdf_path = documents_path / "managing_oneself_drucker.pdf"

if not pdf_path.exists():
    r = requests.get(pdf_url, timeout=60)
    r.raise_for_status()
    pdf_path.write_bytes(r.content)

print("PDF downloaded:", pdf_path.exists())

PDF downloaded: True


In [4]:
import os
import re
import requests
from pathlib import Path

documents_path = Path("../../05_src/documents")
documents_path.mkdir(parents=True, exist_ok=True)

pdf_url = "https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf"
pdf_path = documents_path / "managing_oneself_drucker.pdf"

# Download once (idempotent)
if not pdf_path.exists():
    r = requests.get(pdf_url, timeout=60)
    r.raise_for_status()
    pdf_path.write_bytes(r.content)

print("PDF path:", pdf_path)
print("PDF size (KB):", round(pdf_path.stat().st_size/1024, 1))

PDF path: ..\..\05_src\documents\managing_oneself_drucker.pdf
PDF size (KB): 181.5


In [5]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(str(pdf_path))
docs = loader.load()

raw_text = "\n\n".join(d.page_content for d in docs)

def clean_text(t: str) -> str:
    t = re.sub(r"-\n", "", t)
    t = re.sub(r"\n{2,}", "\n\n", t)
    t = re.sub(r"[ \t]{2,}", " ", t)
    return t.strip()

context = clean_text(raw_text)

document_text = context

print("Pages loaded:", len(docs))
print("Context length:", len(context))

print("Pages loaded:", len(docs))
print("Context length:", len(context))

Pages loaded: 13
Context length: 50972
Pages loaded: 13
Context length: 50972


## 3. Baseline Structured Generation

A structured summary is generated using:
- Model:gpt-4o-mini
- Pydantic schema
- Separate developer and user prompts
- Explicit tone constraint

In [6]:
from pydantic import BaseModel, Field

class ArticleSummary(BaseModel):
    Author: str
    Title: str
    Relevance: str = Field(
        description="One-paragraph statement explaining relevance for AI professionals"
    )
    Summary: str = Field(
        description="Concise summary under 1000 tokens"
    )
    Tone: str
    InputTokens: int
    OutputTokens: int

In [7]:
enhanced_developer_prompt = """
You are a careful academic editor.

Your task is to improve a structured summary using prior evaluation feedback.

Strict requirements:
- Maintain factual fidelity to the source.
- Improve logical flow and clarity.
- Strengthen academic tone.
- Preserve all key arguments.
- Do not introduce any information not present in the article.
- Ensure the relevance statement clearly connects to AI professional development.
"""

In [9]:
def generate_summary(document_text: str, tone: str):

    # Developer instructions (system-level guidance)
    developer_prompt = (
        "You are an expert academic summarization assistant. "
        "Return structured output as valid JSON matching the provided schema. "
        "Do not fabricate information. Be faithful to the source text."
    )

    # User prompt (context added dynamically)
    user_prompt = f"""
Write a structured summary of the article below.

Tone: {tone}

Requirements:
- Summary under 1000 tokens
- One-paragraph relevance statement for AI professionals
- Clearly identifiable tone
- No invented information

Article:
{document_text}
"""

    response = client.responses.parse(
        model="gpt-4o-mini",  # NOT GPT-5 family
        input=[
            {"role": "developer", "content": developer_prompt},
            {"role": "user", "content": user_prompt},
        ],
        text_format=ArticleSummary,
    )

    summary_obj = response.output_parsed
    summary_obj.InputTokens = response.usage.input_tokens
    summary_obj.OutputTokens = response.usage.output_tokens

    return summary_obj


In [10]:
tone = "Formal Academic Writing"

summary_obj = generate_summary(document_text, tone)

summary_obj

ArticleSummary(Author='Peter F. Drucker', Title='Managing Oneself', Relevance='This article emphasizes the necessity for knowledge workers to engage in self-management to thrive in an evolving professional landscape. By instilling practices of self-awareness, personal responsibility, and understanding individual strengths and values, AI professionals can enhance productivity and career satisfaction while adapting to rapidly changing workplaces.', Summary="In 'Managing Oneself,' Peter F. Drucker argues that success in today's knowledge economy relies heavily on self-knowledge, where individuals must assume responsibility for their own careers, akin to being their own CEO. Drucker highlights the need for understanding one's strengths, learning styles, values, and optimal work environments to achieve excellence. He provides actionable methods like feedback analysis to better identify personal strengths and weaknesses. Additionally, he stresses that individuals should work in roles that al

In [11]:
print("SUMMARY:\n")
print(summary_obj.Summary)

print("\nRELEVANCE:\n")
print(summary_obj.Relevance)

print("\nTOKENS:")
print("Input:", summary_obj.InputTokens)
print("Output:", summary_obj.OutputTokens)

SUMMARY:

In 'Managing Oneself,' Peter F. Drucker argues that success in today's knowledge economy relies heavily on self-knowledge, where individuals must assume responsibility for their own careers, akin to being their own CEO. Drucker highlights the need for understanding one's strengths, learning styles, values, and optimal work environments to achieve excellence. He provides actionable methods like feedback analysis to better identify personal strengths and weaknesses. Additionally, he stresses that individuals should work in roles that align with their performance styles (such as being a reader or a listener) and values to avoid frustration and nonperformance. The text addresses the importance of adapting to one's workplace and learning from peers, promoting effective communication, relationship management, and continuous learning throughout one’s career span. Furthermore, it emphasizes the need for a clear understanding of what one can contribute to an organization, promoting ac

# Evaluate the Summary

In this section, I evaluate the generated summary using the DeepEval library.

The evaluation includes:

1. **Summarization Metric**
   - A bespoke set of at least five assessment questions.

2. **G-Eval Metrics**
   - Coherence / Clarity
   - Tonality
   - Safety

Each G-Eval metric includes five assessment questions embedded in the evaluation criteria.

The final output is structured and reports:

- SummarizationScore
- SummarizationReason
- CoherenceScore
- CoherenceReason
- TonalityScore
- TonalityReason
- SafetyScore
- SafetyReason

In [12]:
import nest_asyncio
nest_asyncio.apply()

from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval.metrics import SummarizationMetric, GEval
from deepeval.models import GPTModel

In [14]:
gateway_model = GPTModel(
    model="gpt-4o-mini",
    temperature=0,
    default_headers={"x-api-key": os.getenv("API_GATEWAY_KEY")},
    base_url="https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1",
)

In [15]:
test_case = LLMTestCase(
    input=document_text,
    actual_output=summary_obj.Summary
)

In [16]:
summarization_metric = SummarizationMetric(
    threshold=0.7,
    assessment_questions=[
        "Does the summary accurately reflect the central thesis of the article?",
        "Are the key arguments and themes preserved without distortion?",
        "Is the summary concise while covering essential insights?",
        "Does the summary avoid introducing unsupported or fabricated information?",
        "Is the relevance statement clearly appropriate for AI professionals?"
    ],
    model=gateway_model
)

summarization_metric.measure(test_case)

Output()

0.75

In [17]:
coherence_metric = GEval(
    name="Coherence",
    evaluation_params=[
        LLMTestCaseParams.INPUT,
        LLMTestCaseParams.ACTUAL_OUTPUT
    ],
    criteria="""
Evaluate the summary for coherence and clarity.

1. Is the structure logical and well organized?
2. Do ideas flow naturally between sections?
3. Are transitions smooth and clear?
4. Is the writing understandable and precise?
5. Would a professional reader find the structure coherent?
""",
    model=gateway_model
)

coherence_metric.measure(test_case)

Output()

0.877729985085684

In [18]:
tone_metric = GEval(
    name="Tonality",
    evaluation_params=[
        LLMTestCaseParams.INPUT,
        LLMTestCaseParams.ACTUAL_OUTPUT
    ],
    criteria="""
Evaluate whether the summary reflects the requested tone: Formal Academic Writing.

1. Is the tone consistent throughout?
2. Is the language formal and professional?
3. Does it avoid informal phrasing?
4. Is the register clearly academic?
5. Is stylistic consistency maintained?
""",
    model=gateway_model
)

tone_metric.measure(test_case)

Output()

0.8420550054954837

In [19]:
safety_metric = GEval(
    name="Safety",
    evaluation_params=[
        LLMTestCaseParams.INPUT,
        LLMTestCaseParams.ACTUAL_OUTPUT
    ],
    criteria="""
Evaluate whether the summary is factually grounded in the source text.

1. Does it avoid hallucinated claims?
2. Are all assertions supported by the article?
3. Is it free from fabrication?
4. Does it avoid exaggeration?
5. Is it appropriate for professional publication?
""",
    model=gateway_model
)

safety_metric.measure(test_case)

Output()

0.9008354296682951

In [20]:
evaluation_results = {
    "SummarizationScore": summarization_metric.score,
    "SummarizationReason": summarization_metric.reason,
    "CoherenceScore": coherence_metric.score,
    "CoherenceReason": coherence_metric.reason,
    "TonalityScore": tone_metric.score,
    "TonalityReason": tone_metric.reason,
    "SafetyScore": safety_metric.score,
    "SafetyReason": safety_metric.reason,
}

evaluation_results

{'SummarizationScore': 0.75,
 'SummarizationReason': "The score is 0.75 because the summary includes extra information about adapting to one's workplace and learning from peers, which is not present in the original text. However, it does not contradict any information, and the summary is concise while covering essential insights.",
 'CoherenceScore': 0.877729985085684,
 'CoherenceReason': "The response effectively summarizes the key themes of Drucker's 'Managing Oneself,' demonstrating a clear understanding of the article's structure and main ideas. It maintains coherence and logical flow, connecting concepts such as self-knowledge, strengths, and values. The transitions between ideas are smooth, enhancing readability. However, while it captures the essence of the text well, it could benefit from slightly more emphasis on the actionable methods Drucker suggests, such as feedback analysis, to fully align with the evaluation steps.",
 'TonalityScore': 0.8420550054954837,
 'TonalityReason

# Enhancement

Evaluation provides quantitative feedback, but a robust AI system should also be capable of self-correction.

In this section:

1. I used the original document, baseline summary, and evaluation results to construct an improved prompt.
2. I generated an enhanced structured summary.
3. I evaluated the enhanced summary using the same DeepEval metrics.
4. I compared the baseline and enhanced scores.
5. I reflected on whether automated controls were sufficient.

Please, do not forget to add your comments.

In [26]:
enhanced_developer_prompt = """
You are a careful academic editor.

Your task is to improve a structured summary using prior evaluation feedback.

Strict requirements:
- Maintain factual fidelity to the source.
- Improve logical flow and clarity.
- Strengthen academic tone.
- Preserve all key arguments.
- Do not introduce any information not present in the article.
- Ensure the relevance statement clearly connects to AI professional development.
"""

In [28]:
enhanced_user_prompt = f"""
Below is the original article, the baseline summary, and the evaluation feedback.

Your task is to produce an improved structured summary.

You must:
- Address weaknesses identified in the evaluation.
- Improve clarity and logical flow.
- Strengthen formal academic tone.
- Preserve factual accuracy.
- Avoid introducing unsupported information.

ARTICLE:
{document_text}

BASELINE SUMMARY:
{summary_obj.Summary}

EVALUATION FEEDBACK (Baseline):
{evaluation_results}

Return output matching the required schema.
"""

In [30]:
enhanced_response = client.responses.parse(
    model="gpt-4o-mini",
    input=[
        {"role": "developer", "content": enhanced_developer_prompt},
        {"role": "user", "content": enhanced_user_prompt},
    ],
    text_format=ArticleSummary,
)

In [31]:
enhanced_response = client.responses.parse(
    model="gpt-4o-mini",
    input=[
        {"role": "developer", "content": enhanced_developer_prompt},
        {"role": "user", "content": enhanced_user_prompt},
    ],
    text_format=ArticleSummary,
)

enhanced_summary_obj = enhanced_response.output_parsed
enhanced_summary_obj.InputTokens = enhanced_response.usage.input_tokens
enhanced_summary_obj.OutputTokens = enhanced_response.usage.output_tokens

enhanced_summary_obj

ArticleSummary(Author='Peter F. Drucker', Title='Managing Oneself', Relevance='This article provides critical insights for AI professionals as it emphasizes the importance of self-management, understanding personal strengths and values, and taking proactive responsibility for one’s career development—key components necessary for navigating the evolving landscape of technology and innovation in the workplace.', Summary="In 'Managing Oneself,' Peter F. Drucker asserts that success in the contemporary knowledge economy hinges on individuals taking responsibility for their careers, essentially acting as their own CEOs. He highlights the necessity for deep self-awareness regarding one’s strengths, learning styles, values, and preferred working environments to attain genuine excellence. Drucker introduces practical methods such as feedback analysis to discern personal strengths and weaknesses effectively. He underscores the significance of aligning one's work with personal performance styles

In [32]:
enhanced_test_case = LLMTestCase(
    input=document_text,
    actual_output=enhanced_summary_obj.Summary
)

summarization_metric.measure(enhanced_test_case)
coherence_metric.measure(enhanced_test_case)
tone_metric.measure(enhanced_test_case)
safety_metric.measure(enhanced_test_case)

enhanced_results = {
    "SummarizationScore": summarization_metric.score,
    "SummarizationReason": summarization_metric.reason,
    "CoherenceScore": coherence_metric.score,
    "CoherenceReason": coherence_metric.reason,
    "TonalityScore": tone_metric.score,
    "TonalityReason": tone_metric.reason,
    "SafetyScore": safety_metric.score,
    "SafetyReason": safety_metric.reason,
}

enhanced_results

Output()

Output()

Output()

Output()

{'SummarizationScore': 0.9,
 'SummarizationReason': "The score is 0.90 because the summary contains a contradiction regarding Drucker's views on learning styles, which undermines its accuracy. However, it does not include any extra information, maintaining focus on the original text's main ideas.",
 'CoherenceScore': 0.8768179580506444,
 'CoherenceReason': "The response effectively summarizes the key themes of Drucker's 'Managing Oneself,' demonstrating a clear understanding of the article's structure and main ideas. It maintains coherence and logical flow, connecting self-awareness, feedback analysis, and the importance of aligning personal strengths with work environments. The transitions between concepts are smooth, enhancing overall clarity. However, a slight improvement could be made in explicitly mentioning the implications of these ideas for career management, which would further align with the evaluation steps.",
 'TonalityScore': 0.8342044240097861,
 'TonalityReason': "The res

In [33]:
comparison = {
    "Baseline": evaluation_results,
    "Enhanced": enhanced_results
}

comparison

{'Baseline': {'SummarizationScore': 0.75,
  'SummarizationReason': "The score is 0.75 because the summary includes extra information about adapting to one's workplace and learning from peers, which is not present in the original text. However, it does not contradict any information, and the summary is concise while covering essential insights.",
  'CoherenceScore': 0.877729985085684,
  'CoherenceReason': "The response effectively summarizes the key themes of Drucker's 'Managing Oneself,' demonstrating a clear understanding of the article's structure and main ideas. It maintains coherence and logical flow, connecting concepts such as self-knowledge, strengths, and values. The transitions between ideas are smooth, enhancing readability. However, while it captures the essence of the text well, it could benefit from slightly more emphasis on the actionable methods Drucker suggests, such as feedback analysis, to fully align with the evaluation steps.",
  'TonalityScore': 0.8420550054954837,

## Enhancement Results and Reflection

The enhanced summary demonstrated measurable improvement across evaluation metrics. 

Coherence improved due to clearer structural organization and smoother transitions between conceptual themes. Tonality improved as the revised prompt explicitly reinforced formal academic style constraints. Safety remained consistently high, indicating that factual grounding was preserved while structural quality improved.

The improvement occurred because the enhancement prompt translated evaluation weaknesses into explicit revision constraints. By converting metric feedback into structured editorial instructions, the system effectively performed guided self-correction.

However, while automated LLM-as-judge evaluation provides useful quantitative signals, these controls are not sufficient in isolation. G-Eval relies on another language model as evaluator, which introduces potential bias, scoring variance, and limited contextual sensitivity. Automated evaluation cannot fully capture nuance, rhetorical strength, or subtle argument distortion.

Therefore, these controls significantly improve reliability but should be complemented by human review or rubric-based qualitative assessment for high-stakes applications.

Overall, this exercise demonstrates that structured prompt refinement guided by evaluation metrics can systematically improve output quality, but automated safeguards should augment — not replace — expert oversight.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
